# Notebook 04 — Analyse NLP des messages radio F1

**Objectif** : Analyser les communications radio pilote-ingénieur pour extraire sentiment et type  
**Pipeline** :
1. Récupération des transcriptions via FastF1
2. Nettoyage & tokenisation
3. Sentiment analysis (`cardiffnlp/twitter-roberta-base-sentiment`)
4. Classification du type (alerte technique, stratégie, encouragement, incident)
5. Corrélation sentiment négatif / dégradation de performance


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from src.data_loader import load_session, get_laps_features
from src.nlp import (
    clean_text,
    classify_message_type,
    load_sentiment_pipeline,
    analyze_sentiment_batch,
    build_radio_dataset,
    correlate_sentiment_performance,
    plot_sentiment_distribution,
    plot_driver_sentiment,
    plot_sentiment_performance_correlation,
    HAS_TRANSFORMERS,
    CATEGORY_RULES,
)

plt.rcParams['figure.dpi'] = 120
print(f'HuggingFace Transformers disponible : {HAS_TRANSFORMERS}')

## 1. Chargement de la session

In [ ]:
YEAR = 2025
GP_NAME = 'Monaco'
SESSION_TYPE = 'R'

session = load_session(YEAR, GP_NAME, SESSION_TYPE)
if session is None:
    print(f'  [INFO] {YEAR} {GP_NAME} indisponible — fallback 2024')
    YEAR = 2024
    session = load_session(YEAR, GP_NAME, SESSION_TYPE)
laps_df = get_laps_features(session) if session else pd.DataFrame()
print(f'Tours chargés : {len(laps_df)}')

## 2. Chargement du pipeline de sentiment

In [ ]:
pipe = load_sentiment_pipeline()

# Test rapide
if pipe:
    test_messages = [
        "The car is amazing today, brilliant lap!",
        "We have a hydraulic issue, losing power.",
        "Box this lap, we go to medium tyres.",
    ]
    results = analyze_sentiment_batch(test_messages, pipe)
    for msg, res in zip(test_messages, results):
        print(f'  [{res["label"]:8s} {res["score"]:.2f}] {msg}')

## 3. Construction du dataset radio annoté

In [ ]:
radio_df = build_radio_dataset(session, pipe)
print(f'Messages traités : {len(radio_df)}')
radio_df.head(10)

In [ ]:
print('Répartition des types :')
print(radio_df['Type'].value_counts().to_string())
print('\nRépartition des sentiments :')
print(radio_df['Sentiment'].value_counts().to_string())

## 4. Exploration des règles de classification

In [ ]:
# Démonstration des règles de classification
test_cases = [
    "Box box box, we go to the pit lane",
    "Engine is overheating, we have a problem",
    "Brilliant job, keep pushing!",
    "Safety car on track, take it easy",
    "Roger that, understood",
]

print('Exemples de classification :')
for msg in test_cases:
    cat = classify_message_type(msg)
    clean = clean_text(msg)
    print(f'  [{cat:20s}] {msg}')

## 5. Visualisation — Distribution des sentiments et types

In [ ]:
fig = plot_sentiment_distribution(radio_df)
plt.show()

## 6. Heatmap — Sentiment par pilote et type

In [ ]:
fig = plot_driver_sentiment(radio_df)
plt.show()

## 7. Analyse approfondie par pilote

In [ ]:
if 'Driver' in radio_df.columns:
    driver_stats = radio_df.groupby('Driver').agg(
        NbMessages=('Message', 'count'),
        SentimentMoyen=('SentimentNum', 'mean'),
        PctNegatif=('Sentiment', lambda x: (x == 'negative').mean() * 100),
        PctPositif=('Sentiment', lambda x: (x == 'positive').mean() * 100),
    ).round(2).sort_values('SentimentMoyen')
    
    display(driver_stats)

    # Barplot du sentiment moyen par pilote
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in driver_stats['SentimentMoyen']]
    ax.bar(driver_stats.index, driver_stats['SentimentMoyen'], color=colors, alpha=0.85)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel('Sentiment moyen (−1 à +1)')
    ax.set_title('Sentiment radio moyen par pilote', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 8. Corrélation Sentiment ↔ Performance

In [ ]:
corr_df = correlate_sentiment_performance(radio_df, laps_df)
print(f'Pilotes en commun : {len(corr_df)}')
if not corr_df.empty:
    display(corr_df.sort_values('AvgSentiment'))

In [ ]:
fig = plot_sentiment_performance_correlation(corr_df)
plt.show()

if not corr_df.empty and len(corr_df) > 2:
    r = np.corrcoef(corr_df['AvgSentiment'], corr_df['AvgTimeDelta'])[0, 1]
    print(f'Coefficient de corrélation Pearson : r = {r:.3f}')
    print('Interprétation : un r négatif indique qu\'un sentiment plus négatif est associé à de moins bonnes performances.')

## 9. Messages les plus négatifs

In [ ]:
if 'Sentiment' in radio_df.columns and 'SentimentScore' in radio_df.columns:
    neg_msgs = radio_df[radio_df['Sentiment'] == 'negative'].nlargest(10, 'SentimentScore')
    
    print('Top 10 messages les plus négatifs :')
    for _, row in neg_msgs.iterrows():
        drv = row.get('Driver', '?')
        msg = row.get('CleanMessage', row.get('Message', ''))
        typ = row.get('Type', '?')
        score = row.get('SentimentScore', 0)
        print(f'  [{drv} | {typ:20s} | score: {score:.2f}] {msg[:80]}')

## 10. Rapport de synthèse

In [ ]:
print('=== RAPPORT NLP ===' )
print(f'Session : {YEAR} GP {GP_NAME} {SESSION_TYPE}')
print(f'Messages analysés : {len(radio_df)}')

if not radio_df.empty:
    for sent in ['positive', 'neutral', 'negative']:
        n = (radio_df['Sentiment'] == sent).sum()
        pct = 100 * n / len(radio_df)
        print(f'  {sent:10s}: {n:4d} ({pct:.1f}%)')
    
    print('\nTypes de messages :')
    for typ, cnt in radio_df['Type'].value_counts().items():
        pct = 100 * cnt / len(radio_df)
        print(f'  {typ:22s}: {cnt:4d} ({pct:.1f}%)')

if not corr_df.empty and len(corr_df) > 2:
    r = np.corrcoef(corr_df['AvgSentiment'], corr_df['AvgTimeDelta'])[0, 1]
    print(f'\nCorrélation sentiment/performance : r = {r:.3f}')